# 05 — Watchlist Predictions

Predicts a rating for every unseen film on the watchlist, for the Streamlit app.

**Input:** the Letterboxd export (`watchlist.csv`), plus `data/interim/viewings.csv` and
`films_enriched.csv` for the rated history
**Output:** watchlist predictions, one row per film

The watchlist goes through the same pipeline as the rated films: matched with the same
rules (`src/tmdb.py`), given the same features, and scored by Model 4 RF refitted on all
1,192 rated viewings. Watchlist API responses are cached in **separate files**, so the
rated-film caches from `02` are never touched.

## 1. Load the watchlist

In [15]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from src.letterboxd import load_export, film_key
from src.tmdb import make_headers, match_all, AUTO_ACCEPT, search_film, fetch_all_details

load_dotenv()
export = load_export(os.getenv("LETTERBOXD_EXPORT_DIR"))

watchlist = export["watchlist"].copy()
watchlist["film_key"] = film_key(watchlist)

rated   = set(film_key(export["ratings"]))
watched = set(film_key(export["watched"]))

print(f"watchlist rows   : {len(watchlist)}")
print(f"unique film_keys : {watchlist['film_key'].nunique()}")
print(f"missing year     : {watchlist['Year'].isna().sum()}")
print(f"already rated    : {watchlist['film_key'].isin(rated).sum()}")
print(f"already watched  : {watchlist['film_key'].isin(watched).sum()}")

watchlist rows   : 4031
unique film_keys : 4022
missing year     : 8
already rated    : 0
already watched  : 0


In [3]:
dupes = watchlist[watchlist["film_key"].duplicated(keep=False)].sort_values("film_key")
print(f"{len(dupes)} rows share a film_key ({dupes['film_key'].nunique()} keys)\n")
print(dupes[["Name", "Year", "Letterboxd URI"]].to_string(index=False))

print("\nmissing year:\n")
print(watchlist[watchlist["Year"].isna()][["Name", "Letterboxd URI"]].to_string(index=False))

10 rows share a film_key (1 keys)

                    Name   Year        Letterboxd URI
              The Castle 1997.0  https://boxd.it/1sWI
              The Castle 1997.0  https://boxd.it/1Pho
                 Polaris    NaN  https://boxd.it/vMS2
       The Memory Police    NaN  https://boxd.it/scb6
         The Governesses    NaN  https://boxd.it/AdVq
           Tower Stories    NaN  https://boxd.it/nzTg
              Love Child    NaN  https://boxd.it/fAP4
The Bookie & the Bruiser    NaN  https://boxd.it/N6Oe
    Here Comes the Flood    NaN  https://boxd.it/qdjm
              Lily May B    NaN https://boxd.it/13ssm

missing year:

                    Name        Letterboxd URI
                 Polaris  https://boxd.it/vMS2
       The Memory Police  https://boxd.it/scb6
         The Governesses  https://boxd.it/AdVq
           Tower Stories  https://boxd.it/nzTg
              Love Child  https://boxd.it/fAP4
The Bookie & the Bruiser  https://boxd.it/N6Oe
    Here Comes the Flood  

In [4]:
undated  = watchlist["film_key"].isna()
collides = watchlist["film_key"].duplicated(keep=False) & ~undated

collisions = watchlist[collides].copy()

films_wl = (watchlist[~undated & ~collides]
            .rename(columns={"Name": "film_title", "Year": "film_year",
                             "Letterboxd URI": "film_uri"})
            [["film_key", "film_title", "film_year", "film_uri"]]
            .reset_index(drop=True))

print(f"excluded, no year      : {undated.sum()}")
print(f"held back, collision   : {collides.sum()} rows, {collisions['film_key'].nunique()} key(s)")
print(f"to match automatically : {len(films_wl)}")

excluded, no year      : 8
held back, collision   : 2 rows, 1 key(s)
to match automatically : 4021


In [6]:
HEADERS = make_headers(os.getenv("TMDB_TOKEN"))

WL_SEARCH_CACHE = Path("data/cache/watchlist_search_raw.json")

wl_matches = match_all(films_wl, headers=HEADERS, cache_path=WL_SEARCH_CACHE)

  100/4021 processed (100 API calls)
  200/4021 processed (200 API calls)
  300/4021 processed (300 API calls)
  400/4021 processed (400 API calls)
  500/4021 processed (500 API calls)
  600/4021 processed (600 API calls)
  700/4021 processed (700 API calls)
  800/4021 processed (800 API calls)
  900/4021 processed (900 API calls)
  1000/4021 processed (1000 API calls)
  1100/4021 processed (1100 API calls)
  1200/4021 processed (1200 API calls)
  1300/4021 processed (1300 API calls)
  1400/4021 processed (1400 API calls)
  1500/4021 processed (1500 API calls)
  1600/4021 processed (1600 API calls)
  1700/4021 processed (1700 API calls)
  1800/4021 processed (1800 API calls)
  1900/4021 processed (1900 API calls)
  2000/4021 processed (2000 API calls)
  2100/4021 processed (2100 API calls)
  2200/4021 processed (2200 API calls)
  2300/4021 processed (2300 API calls)
  2400/4021 processed (2400 API calls)
  2500/4021 processed (2500 API calls)
  2600/4021 processed (2600 API calls)
  27

In [8]:
print(f"films: {len(wl_matches)}\n")
print("confidence breakdown")
for tier, n in wl_matches["confidence"].value_counts().items():
    print(f"  {tier:12s} {n:5d}  ({n/len(wl_matches)*100:5.1f}%)")

wl_review = wl_matches[~wl_matches["confidence"].isin(AUTO_ACCEPT)]
print(f"\nauto-accepted  : {len(wl_matches) - len(wl_review)} "
      f"({(len(wl_matches) - len(wl_review)) / len(wl_matches) * 100:.1f}%)")
print(f"needs review   : {len(wl_review)}")
print(f"no match at all: {wl_matches['tmdb_id'].isna().sum()}")

films: 4021

confidence breakdown
  exact         3673  ( 91.3%)
  year_off       295  (  7.3%)
  weak            49  (  1.2%)
  close            3  (  0.1%)
  no_match         1  (  0.0%)

auto-accepted  : 3968 (98.7%)
needs review   : 53
no match at all: 1


In [9]:
print(wl_review.sort_values(["confidence", "similarity"])[
    ["film_title", "film_year", "matched_title", "matched_year",
     "confidence", "similarity", "vote_count"]
].to_string(index=False))

                                             film_title  film_year                                           matched_title  matched_year confidence  similarity  vote_count
                                                Monster     2018.0                                                 Monster        2018.0      close       1.000         1.0
                                                  Alpha     2025.0                                                   Alpha        2025.0      close       1.000       140.0
                                                Solaris     2007.0                                                 Solaris        2007.0      close       1.000         1.0
   Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles     1975.0                                                     NaN           NaN   no_match       0.000         NaN
                                  Q: The Winged Serpent     1982.0                                                       Q        1982.0    

In [11]:
lookups = {
    "Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)": "Jeanne Dielman",
    "Precious (2009)":                           "Precious",
    "The Ear (1970)":                            "The Ear",
    "The Night (2021)":                          "The Night",
    "Monster (2018)":                            "Monster",
    "Nausicaä of the Valley of the Wind (1984)": "Nausicaä of the Valley of the Wind",
    "The Murmuring (2022)":                      "The Murmuring",
    "The Civil War on Drugs (2011)":             "The Civil War on Drugs",
    "Sybil (1976)":                              "Sybil",
    "The Castle (1997)":                         "The Castle",
}

current = wl_matches.set_index("film_key")["tmdb_id"]

for key, query in lookups.items():
    year = int(key[-5:-1])
    print(f"=== {key}   current match: {current.get(key, 'held back')}")
    for r in search_film(query, headers=HEADERS):
        release = r.get("release_date") or ""
        y = int(release[:4]) if release[:4].isdigit() else None
        if y is None or abs(y - year) <= 5:
            print(f"  {r['id']:>8}  {y or '????'}  {r.get('vote_count', 0):>6} votes  "
                  f"{r['title']}  /  {r['original_title']}")
    print()

=== Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)   current match: nan
    308191  1975       2 votes  Around Jeanne Dielman  /  Autour de Jeanne Dielman
     44012  1976     410 votes  Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles  /  Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles
   1404470  ????       0 votes  Exercises in Style. Jeanne Dielman  /  Exercises in Style. Jeanne Dielman

=== Precious (2009)   current match: 1772671.0
     25793  2009    1821 votes  Precious: Based on the Novel 'Push' by Sapphire  /  Precious: Based on the Novel 'Push' by Sapphire
   1010417  ????       0 votes  Precious Cargo  /  Precious Cargo

=== The Ear (1970)   current match: 888896.0
    888896  1970       0 votes  The Burning Ear  /  The Burning Ear
    340000  1970       1 votes  The Eye Hears, the Ear Sees  /  The Eye Hears, the Ear Sees
    799285  ????       0 votes  Into the Ear, Everybody  /  Into the Ear, Everybody

=== The Night (2021)   current match: 604360.0


In [12]:
second = [
    ("Ucho",      None),   # The Ear's original Czech title
    ("The Night", 2020),
    ("The Night", 2021),
    ("Monster",   2021),
    ("Monster",   2018),
]

for query, year in second:
    print(f"=== {query!r}, year={year}")
    for r in search_film(query, year, headers=HEADERS)[:8]:
        release = r.get("release_date") or "????"
        print(f"  {r['id']:>8}  {release[:4]}  {r.get('vote_count', 0):>6} votes  "
              f"{r['title']}  /  {r['original_title']}")
    print()

=== 'Ucho', year=None
     88953  1990      74 votes  The Ear  /  Ucho
    334772  1945       8 votes  The Eye & the Ear  /  Oko I Ucho
   1276412  2016       0 votes  The Internal Ear  /  Ucho wewnętrzne
   1064567  1972       0 votes  Kým sa ucho neodbije  /  Kým sa ucho neodbije
     64711  1949     142 votes  Long-Haired Hare  /  Long-Haired Hare
     63899  1977      57 votes  White Bim Black Ear  /  Белый Бим Чёрное ухо
    588336  2016       2 votes  The Mystery of Van Gogh's Ear  /  The Mystery of Van Gogh's Ear
     19316  2004     275 votes  Saving Face  /  Saving Face

=== 'The Night', year=2020
      3112  1955    1905 votes  The Night of the Hunter  /  The Night of the Hunter
    547565  2021    1392 votes  The Night House  /  The Night House
    565743  2019    1239 votes  The Vast of Night  /  The Vast of Night
    526007  2020     932 votes  The Night Clerk  /  The Night Clerk
    640796  2020       3 votes  Into the Night  /  Into the Night
    686245  2020     266 vot

In [13]:
WL_OVERRIDES = {
    "Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)": 44012,
    "Precious (2009)":  25793,
    "The Ear (1970)":   88953,
    "The Night (2021)": 854531,
    "Monster (2018)":   489932,
}

WL_EXCLUDE = {
    "The New Pope (2020)":          "tv",
    "The Young Pope (2016)":        "tv",
    "Ren Faire (2024)":             "tv",
    "Storm of the Century (1999)":  "tv",
    "Berlin Alexanderplatz (1980)": "tv",
    "Dekalog (1989)":               "tv",
    "Salem's Lot (1979)":           "tv",
    "Little Women (2017)":          "tv",
    "Catch-22 (2019)":              "tv",
    "Sybil (1976)":                 "tv",
    "The Murmuring (2022)":         "not on TMDB",
}

COLLISION_OVERRIDES = {                 # by URI: film_key can't tell these apart
    "https://boxd.it/1sWI": 26891,      # The Castle — Haneke (Das Schloß)
    "https://boxd.it/1Pho": 13852,      # The Castle — Sitch
}

wl_final = wl_matches.merge(films_wl[["film_key", "film_uri"]], on="film_key", how="left")

for key in [*WL_OVERRIDES, *WL_EXCLUDE]:
    if key not in set(wl_final["film_key"]):
        print(f"warning: key not found — {key!r}")

for key, tmdb_id in WL_OVERRIDES.items():
    mask = wl_final["film_key"] == key
    wl_final.loc[mask, "tmdb_id"] = tmdb_id
    wl_final.loc[mask, "confidence"] = "manual_override"

before = len(wl_final)
wl_final = wl_final[~wl_final["film_key"].isin(WL_EXCLUDE)]
n_excluded = before - len(wl_final)

castle = pd.DataFrame({
    "film_key":   collisions["film_key"].values,
    "film_title": collisions["Name"].values,
    "film_year":  collisions["Year"].values,
    "film_uri":   collisions["Letterboxd URI"].values,
})
castle["tmdb_id"]    = castle["film_uri"].map(COLLISION_OVERRIDES)
castle["confidence"] = "manual_override"
assert castle["tmdb_id"].notna().all(), "a collision URI has no override"

wl_final = pd.concat([wl_final, castle], ignore_index=True)

print(f"overrides applied : {len(WL_OVERRIDES)} + {len(castle)} by URI")
print(f"excluded          : {n_excluded}")
print(f"final             : {len(wl_final)} films\n")
print(wl_final["confidence"].value_counts())
print(f"\nmissing tmdb_id   : {wl_final['tmdb_id'].isna().sum()}")
print(f"duplicate tmdb_id : {wl_final['tmdb_id'].duplicated().sum()}")

overrides applied : 5 + 2 by URI
excluded          : 11
final             : 4012 films

confidence
exact              3673
year_off            295
weak                 35
manual_override       7
close                 2
Name: count, dtype: int64

missing tmdb_id   : 0
duplicate tmdb_id : 0


In [14]:
wl_final[wl_final["confidence"] == "year_off"][
    ["film_title", "film_year", "matched_title", "matched_year", "vote_count"]
].sample(10, random_state=0).to_string(index=False)

"                 film_title  film_year               matched_title  matched_year  vote_count\n                 The Bronze     2015.0                  The Bronze        2016.0       404.0\n       The Celluloid Closet     1995.0        The Celluloid Closet        1996.0       120.0\n          Heaven Knows What     2014.0           Heaven Knows What        2015.0       204.0\nAt the First Breath of Wind     2002.0 At the First Breath of Wind        2003.0        15.0\n            In Vanda's Room     2000.0             In Vanda's Room        2001.0        57.0\n             White Material     2009.0              White Material        2010.0       181.0\n                  Ned Rifle     2014.0                   Ned Rifle        2015.0        58.0\n            The Daytrippers     1996.0             The Daytrippers        1997.0       113.0\n           Imagine Me & You     2005.0            Imagine Me & You        2006.0      1136.0\n            35 Shots of Rum     2008.0             35 Shots

In [16]:
WL_DETAILS_CACHE = Path("data/cache/watchlist_details.json")

wl_details = fetch_all_details(wl_final["tmdb_id"], headers=HEADERS, cache_path=WL_DETAILS_CACHE)
print(wl_details.shape)

  100/4012 processed (100 API calls)
  200/4012 processed (200 API calls)
  300/4012 processed (300 API calls)
  400/4012 processed (400 API calls)
  500/4012 processed (500 API calls)
  600/4012 processed (600 API calls)
  700/4012 processed (700 API calls)
  800/4012 processed (800 API calls)
  900/4012 processed (900 API calls)
  1000/4012 processed (1000 API calls)
  1100/4012 processed (1100 API calls)
  1200/4012 processed (1200 API calls)
  1300/4012 processed (1300 API calls)
  1400/4012 processed (1400 API calls)
  1500/4012 processed (1500 API calls)
  1600/4012 processed (1600 API calls)
  1700/4012 processed (1700 API calls)
  1800/4012 processed (1800 API calls)
  1900/4012 processed (1900 API calls)
  2000/4012 processed (2000 API calls)
  2100/4012 processed (2100 API calls)
  2200/4012 processed (2200 API calls)
  2300/4012 processed (2300 API calls)
  2400/4012 processed (2400 API calls)
  2500/4012 processed (2500 API calls)
  2600/4012 processed (2600 API calls)
  27